# Évaluation de la génération RAG avec Ollama

Ce notebook évalue la couche de génération locale utilisée après le retrieval RAG.

Objectifs :

- tester la génération locale avec Ollama ;
- vérifier que le LLM reformule un brouillon sûr au lieu d'inventer une procédure ;
- mesurer les latences de retrieval et de génération ;
- vérifier le garde-fou anti-hallucination ;
- documenter le fallback déterministe utilisé lorsque la réponse LLM est rejetée.

Dans l'architecture du prototype, Ollama n'est pas utilisé comme source de vérité. Le LLM sert uniquement à reformuler une réponse construite à partir des chunks RAG et d'un brouillon contrôlé.

## 1. Initialisation

Le notebook doit être exécuté avec le kernel du venv du projet : `Amen Bank Chatbot (.venv)`.

Avant de lancer les cellules, vérifier qu'Ollama est disponible localement :

```bash
ollama list
curl http://127.0.0.1:11434/api/tags
```

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

## 2. Imports et configuration

In [ ]:
import json
import time
from typing import Any

import pandas as pd
import requests

from backend.app.core.config import (
    OLLAMA_BASE_URL,
    OLLAMA_ENABLED,
    OLLAMA_MODEL,
    OLLAMA_TIMEOUT_SECONDS,
)
from backend.app.rag.ollama_answerer import OllamaAnswerer
from backend.app.rag.retriever import search
from backend.app.services.chat_service import ChatService

print("OLLAMA_ENABLED:", OLLAMA_ENABLED)
print("OLLAMA_BASE_URL:", OLLAMA_BASE_URL)
print("OLLAMA_MODEL:", OLLAMA_MODEL)
print("OLLAMA_TIMEOUT_SECONDS:", OLLAMA_TIMEOUT_SECONDS)

## 3. Vérification de disponibilité d'Ollama

In [ ]:
def check_ollama_available() -> tuple[bool, str]:
    try:
        response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        response.raise_for_status()
        return True, response.text[:500]
    except requests.exceptions.RequestException as error:
        return False, str(error)


ollama_available, ollama_status = check_ollama_available()

print("Ollama disponible:", ollama_available)
print("Statut:", ollama_status)

## 4. Jeu de questions

Les questions choisies couvrent les principaux cas RAG du prototype : opposition carte, confirmation, chéquier, document, virement et services disponibles.

In [ ]:
test_cases = [
    {
        "question": "Comment faire opposition à une carte ?",
        "category": "carte",
    },
    {
        "question": "Quelles actions nécessitent une confirmation ?",
        "category": "securite",
    },
    {
        "question": "Comment commander un chéquier ?",
        "category": "service",
    },
    {
        "question": "Comment demander un relevé de compte ?",
        "category": "service",
    },
    {
        "question": "Comment faire un virement ?",
        "category": "virement",
    },
    {
        "question": "Quels services sont disponibles sur AMENet ?",
        "category": "amenet",
    },
]

len(test_cases)

## 5. Fonctions utilitaires

On reproduit explicitement le passage par Ollama pour pouvoir observer la réponse brute, puis appliquer le garde-fou du backend.

In [ ]:
answerer = OllamaAnswerer()
chat_service = ChatService()


def call_ollama_for_answer(query: str, results: list[Any]) -> tuple[str | None, str | None, float, str | None]:
    if not ollama_available:
        return None, None, 0.0, "ollama_unavailable"

    context = answerer._build_context(results)
    safe_draft = answerer._build_safe_draft(query=query, results=results)

    start_time = time.perf_counter()

    try:
        response = requests.post(
            f"{OLLAMA_BASE_URL}/api/chat",
            json={
                "model": OLLAMA_MODEL,
                "messages": [
                    {
                        "role": "system",
                        "content": answerer._build_system_prompt(),
                    },
                    {
                        "role": "user",
                        "content": answerer._build_user_prompt(
                            query=query,
                            safe_draft=safe_draft,
                            context=context,
                        ),
                    },
                ],
                "stream": False,
                "options": {
                    "temperature": 0.0,
                    "top_p": 0.7,
                    "num_predict": 260,
                },
            },
            timeout=(3, OLLAMA_TIMEOUT_SECONDS),
        )
        elapsed_ms = round((time.perf_counter() - start_time) * 1000, 2)
        response.raise_for_status()

        payload = response.json()
        raw_answer = payload.get("message", {}).get("content", "").strip()
        processed_answer = answerer._postprocess_answer(raw_answer)

        return raw_answer, processed_answer, elapsed_ms, None

    except requests.exceptions.RequestException as error:
        elapsed_ms = round((time.perf_counter() - start_time) * 1000, 2)
        return None, None, elapsed_ms, str(error)


def get_fallback_answer(query: str, results: list[Any]) -> str:
    return chat_service._build_rag_message(results, query)


print("Fonctions utilitaires prêtes.")

In [ ]:
# Warm-up pour éviter que le premier cas mesure le chargement initial
# du modèle d'embeddings et du modèle Ollama.

warmup_query = "Comment faire opposition à une carte ?"

print("Warm-up retrieval...")
warmup_results = search(warmup_query, top_k=3)

print("Warm-up Ollama...")
_ = answerer.generate_answer(
    query=warmup_query,
    results=warmup_results,
)

print("Warm-up terminé.")

## 6. Exécution de l'évaluation

Pour chaque question :

1. on récupère les chunks avec le retriever RAG ;
2. on construit un brouillon sûr ;
3. on demande à Ollama de reformuler ;
4. on applique le garde-fou ;
5. si la réponse est rejetée, on utilise le fallback déterministe.

In [ ]:
rows = []
top_k = 3

for index, case in enumerate(test_cases, start=1):
    question = case["question"]

    retrieval_start = time.perf_counter()
    results = search(question, top_k=top_k)
    retrieval_ms = round((time.perf_counter() - retrieval_start) * 1000, 2)

    top1 = results[0] if results else None
    safe_draft = answerer._build_safe_draft(query=question, results=results)
    fallback_answer = get_fallback_answer(question, results)

    raw_answer, processed_answer, generation_ms, generation_error = call_ollama_for_answer(
        query=question,
        results=results,
    )

    accepted_by_guardrail = False

    if processed_answer:
        accepted_by_guardrail = answerer._is_safe_grounded_answer(
            query=question,
            answer=processed_answer,
        )

    if processed_answer and accepted_by_guardrail:
        final_answer = processed_answer
        final_answer_source = "ollama"
    else:
        final_answer = fallback_answer
        final_answer_source = "fallback"

    rows.append(
        {
            "id": index,
            "category": case["category"],
            "question": question,
            "top1_title": top1.title if top1 else None,
            "top1_score": round(top1.score, 3) if top1 and top1.score is not None else None,
            "sources_count": len(results),
            "retrieval_ms": retrieval_ms,
            "generation_ms": generation_ms,
            "generation_error": generation_error,
            "ollama_raw_answer": raw_answer,
            "ollama_processed_answer": processed_answer,
            "accepted_by_guardrail": accepted_by_guardrail,
            "final_answer_source": final_answer_source,
            "safe_draft": safe_draft,
            "final_answer": final_answer,
            "final_answer_preview": final_answer[:260] if final_answer else None,
        }
    )

df = pd.DataFrame(rows)
df

## 7. Résumé global

In [ ]:
total = len(df)
accepted = int(df["accepted_by_guardrail"].sum())
fallbacks = int((df["final_answer_source"] == "fallback").sum())

summary = pd.DataFrame(
    [
        {"metric": "Nombre de questions", "value": total},
        {"metric": "Réponses Ollama acceptées", "value": accepted},
        {"metric": "Réponses fallback", "value": fallbacks},
        {"metric": "Taux acceptation Ollama", "value": round(accepted / total, 3) if total else 0},
        {"metric": "Latence moyenne retrieval ms", "value": round(df["retrieval_ms"].mean(), 2)},
        {"metric": "Latence moyenne génération ms", "value": round(df["generation_ms"].mean(), 2)},
        {"metric": "Score top-1 moyen", "value": round(df["top1_score"].mean(), 3)},
    ]
)

summary

## 8. Analyse par question

In [ ]:
df[
    [
        "question",
        "top1_title",
        "top1_score",
        "accepted_by_guardrail",
        "final_answer_source",
        "retrieval_ms",
        "generation_ms",
        "final_answer_preview",
    ]
]

## 9. Réponses rejetées par le garde-fou

Cette section permet d'observer les cas où le LLM local a produit une réponse jugée risquée ou non conforme, puis remplacée par le fallback déterministe.

In [ ]:
rejected = df[df["final_answer_source"] == "fallback"]

if rejected.empty:
    print("Aucune réponse Ollama n'a été rejetée par le garde-fou.")
else:
    display(
        rejected[
            [
                "question",
                "ollama_processed_answer",
                "final_answer",
            ]
        ]
    )

## 10. Exemple détaillé

On affiche un exemple complet : question, brouillon sûr, réponse Ollama, statut du garde-fou et réponse finale.

In [ ]:
example = df.iloc[0]

print("Question :")
print(example["question"])

print("\n--- Brouillon sûr ---")
print(example["safe_draft"])

print("\n--- Réponse Ollama brute / post-traitée ---")
print(example["ollama_processed_answer"])

print("\nAcceptée par le garde-fou :", example["accepted_by_guardrail"])
print("Source de la réponse finale :", example["final_answer_source"])

print("\n--- Réponse finale ---")
print(example["final_answer"])

## 11. Export des résultats

Les résultats sont exportés au format CSV, JSON et Markdown pour le rapport.

In [ ]:
evaluation_dir = PROJECT_ROOT / "evaluation"
evaluation_dir.mkdir(exist_ok=True)

csv_path = evaluation_dir / "ollama_rag_generation_evaluation.csv"
json_path = evaluation_dir / "ollama_rag_generation_evaluation.json"
md_path = evaluation_dir / "ollama_rag_generation_evaluation.md"

df.to_csv(csv_path, index=False)
df.to_json(json_path, orient="records", indent=2, force_ascii=False)

report_columns = [
    "question",
    "top1_title",
    "top1_score",
    "accepted_by_guardrail",
    "final_answer_source",
    "retrieval_ms",
    "generation_ms",
]

markdown_report = "# Résultats de l'évaluation Ollama RAG generation\n\n"
markdown_report += summary.to_markdown(index=False)
markdown_report += "\n\n## Détail par question\n\n"
markdown_report += df[report_columns].to_markdown(index=False)
markdown_report += "\n\n## Exemple détaillé\n\n"
markdown_report += f"**Question :** {example['question']}\n\n"
markdown_report += "**Brouillon sûr :**\n\n"
markdown_report += f"{example['safe_draft']}\n\n"
markdown_report += "**Réponse finale :**\n\n"
markdown_report += f"{example['final_answer']}\n"

md_path.write_text(markdown_report, encoding="utf-8")

print(f"Résultats CSV : {csv_path}")
print(f"Résultats JSON : {json_path}")
print(f"Rapport Markdown : {md_path}")

## 12. Conclusion

Cette évaluation montre que la génération locale n'est pas utilisée comme source autonome de connaissances. Le LLM local intervient après le retrieval RAG et reformule un brouillon contrôlé.

Le garde-fou permet de rejeter les réponses qui sortent du périmètre ou qui demandent des informations sensibles. Dans ce cas, le chatbot revient à une réponse déterministe plus sûre.